# EIA-930 ERCOT Data Pull

| | |
|---|---|
| **Title** | DSC 450 Project 2: EIA-930 ERCOT Data Pull |
| **Author** | Hali Sarah Parsons |
| **Date** | 2026-10-09 |
| **Modified By** | Claude (initial draft) |
| **Description** | Downloads hourly ERCOT demand, day-ahead demand forecast, net generation, and interchange from the EIA Open Data API (Form EIA-930), reshapes it to one row per hour, and saves CSVs for the forecasting notebook. |

## 1. Setup

**Why the API instead of the bulk files:** the API lets us request only ERCOT and only the four series we need, so the pull is small and repeatable.

**Why the key comes from the environment:** this notebook will live on GitHub for the portfolio, and a hard-coded key would be exposed. Get a free key at https://www.eia.gov/opendata/register.php, then either set `EIA_API_KEY` or paste it when prompted.

**Why these dates:** several full years are needed so the models see every season more than once. Change `START` and `END` if the team wants a different window.

In [ ]:
# Title: DSC 450 Project 2 - EIA-930 ERCOT Data Pull
# Author: Hali Sarah Parsons
# Date: 2026-10-09
# Modified By: Claude (initial draft)
# Description: Downloads hourly ERCOT grid data from the EIA Open Data API.

import os
import time
from getpass import getpass

import pandas as pd
import requests

# WHY os.environ.get first: keeps the key out of the saved notebook.
# WHY getpass as fallback: hides the key on screen if it has to be typed.
API_KEY = os.environ.get('EIA_API_KEY') or getpass('EIA API key: ')

BASE_URL = 'https://api.eia.gov/v2/electricity/rto/region-data/data/'
RESPONDENT = 'ERCO'  # EIA's code for ERCOT
# D = demand, DF = day-ahead demand forecast, NG = net generation, TI = total interchange
SERIES_TYPES = ['D', 'DF', 'NG', 'TI']
START = '2022-01-01T00'  # periods are in UTC
END = '2026-09-30T23'
PAGE_SIZE = 5000  # EIA returns at most 5,000 rows per request

## 2. Fetch function

**Why paging:** the full window is roughly 165,000 rows, far over the 5,000-row cap, so the data has to be requested one page at a time.

**Why sort on both period and type:** paging only works if the row order is identical on every request. Four series share each hour, so sorting on period alone could shuffle ties between pages and silently drop or duplicate rows.

**Why retry:** a pull of 30+ requests will occasionally hit a timeout or rate limit. Retrying with a pause avoids restarting the whole download for one bad response.

In [ ]:
def fetch_page(offset):
    '''Return (rows, total_row_count) for one page starting at `offset`.

    WHY a function: the same request is repeated for every page, and
    isolating it keeps the retry logic in one place.
    '''
    params = {
        'api_key': API_KEY,
        'frequency': 'hourly',
        'data[0]': 'value',
        'facets[respondent][]': RESPONDENT,
        'facets[type][]': SERIES_TYPES,
        'start': START,
        'end': END,
        'sort[0][column]': 'period',
        'sort[0][direction]': 'asc',
        'sort[1][column]': 'type',
        'sort[1][direction]': 'asc',
        'offset': offset,
        'length': PAGE_SIZE,
    }
    for attempt in range(3):
        # WHY try/except: network and rate-limit errors are temporary, so
        # they should trigger a retry rather than kill the whole pull.
        try:
            # WHY timeout: without it a stalled connection hangs forever.
            resp = requests.get(BASE_URL, params=params, timeout=60)
            # WHY raise_for_status: turns a 4xx/5xx reply into an error here
            # instead of a confusing KeyError later.
            resp.raise_for_status()
            body = resp.json()['response']
            # WHY int(): the API returns the total as a string.
            return body['data'], int(body['total'])
        except (requests.RequestException, KeyError, ValueError) as err:
            if attempt == 2:
                raise  # WHY: after 3 tries the problem is not temporary
            print(f'Retrying offset {offset} after error: {err}')
            # WHY growing sleep: gives a rate limit time to reset.
            time.sleep(5 * (attempt + 1))

## 3. Run the pull

Loop until the offset passes the total the API reports. The short pause between pages is there to stay polite with a free public API.

In [ ]:
rows = []
offset = 0
total = None

# WHY `total is None`: the total is unknown until the first response.
while total is None or offset < total:
    page, total = fetch_page(offset)
    if not page:
        break  # WHY: guards against an endless loop if the API returns nothing
    rows.extend(page)
    offset += PAGE_SIZE
    print(f'{len(rows):,} of {total:,} rows')
    time.sleep(0.5)

raw = pd.DataFrame(rows)
raw.head()

## 4. Reshape and save

**Why pivot:** the API returns one row per hour *per series* (long format). Forecasting needs one row per hour with each series as a column.

**Why keep UTC as the index and add Central as a column:** UTC has no daylight-saving gaps or repeated hours, so it is the safe key. Central time is what drives human behavior (the 5 pm peak), so it is what the hour-of-day features should use.

**Why save the raw file too:** if a cleaning choice turns out to be wrong, we can redo it without pulling again.

In [ ]:
# WHY to_numeric with coerce: values arrive as strings, and any bad entry
# should become NaN so it shows up in the quality check instead of crashing.
raw['value'] = pd.to_numeric(raw['value'], errors='coerce')

# WHY explicit format: EIA periods look like '2022-01-01T00', which pandas
# may misread without being told the pattern.
raw['period_utc'] = pd.to_datetime(raw['period'], format='%Y-%m-%dT%H', utc=True)

# WHY pivot_table with aggfunc='first': plain pivot errors out if the API
# ever returns a duplicate hour; this keeps one value and moves on.
wide = (
    raw.pivot_table(index='period_utc', columns='type', values='value', aggfunc='first')
    .rename(columns={
        'D': 'demand_mwh',
        'DF': 'demand_forecast_mwh',
        'NG': 'net_generation_mwh',
        'TI': 'interchange_mwh',
    })
    .sort_index()
)
wide.columns.name = None
wide['period_central'] = wide.index.tz_convert('America/Chicago')

raw.to_csv('eia930_erco_raw.csv', index=False)
wide.to_csv('eia930_erco_hourly.csv')
wide.head()

## 5. Quality checks

**Why check before modeling:** time-series models assume an unbroken hourly sequence. Missing hours, nulls, and zero or negative demand readings need to be found now so the cleaning step in the next notebook is a decision, not a surprise. These counts also go in the Methods section of the report.

In [ ]:
# WHY build the expected index: comparing against a complete hourly range
# is the only way to see hours the API left out entirely.
expected = pd.date_range(wide.index.min(), wide.index.max(), freq='h')
missing_hours = expected.difference(wide.index)

print(f'Rows: {len(wide):,}')
print(f'Range: {wide.index.min()} to {wide.index.max()}')
print(f'Missing hours: {len(missing_hours):,}')
print('\nNulls per column:')
print(wide.isna().sum())
# WHY flag <= 0: real grid demand is never zero, so these are reporting errors.
print(f"\nDemand <= 0: {(wide['demand_mwh'] <= 0).sum():,}")

wide.describe()